# 02 — Dataset Visualisation

**Goal.** Qualitatively inspect the recordings before landmark extraction, looking at the conditions that affect MediaPipe tracking.

**Approach**
- Sample up to 2 clips per class (`random_state=42`), giving 400 clips, and plot each clip's container-midpoint frame in a grid.
- Show 8 evenly spaced frames from one random clip to illustrate how a sign unfolds over time.
- Lighting: a histogram of the mean grey-level brightness of the sampled midpoint frames.
- Resolution: count width × height over all 6,000 clips.
- Create an empty annotation template (lighting, background, camera angle, occlusion, hands, signer, comments) for manual review.

**Observations.** Indoor lighting, plain backgrounds, frontal camera, upper body visible, mostly two-handed signing, minimal occlusion and low motion blur, with clear variation between signers. The data suits landmark-based recognition.

**Why it matters.** The inspection is qualitative only and is not used to estimate how often any condition occurs. It motivated the presence masks and gap filling in 05, and the measured detection rates come from 03.

**Outputs.** `random_dataset_samples.png`, `dataset_visual_annotation.csv`, plus the motion, lighting and resolution plots.

In [ ]:
# 1) Import Libraries

import os
import cv2
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from tqdm.auto import tqdm

In [ ]:
# 2) Dataset Path

DATASET_PATH = "/kaggle/input/datasets/shahriar55/signbd-dataset/SignBD-Word"

In [ ]:
# 3) Collect All Videos

VIDEO_EXTENSIONS = (
    ".mp4",
    ".avi",
    ".mov",
    ".mkv",
    ".webm"
)

video_paths = []

for root, dirs, files in os.walk(DATASET_PATH):

    for file in files:

        if file.lower().endswith(VIDEO_EXTENSIONS):

            video_paths.append(
                os.path.join(root,file)
            )

print("Total Videos:",len(video_paths))

In [ ]:
# 4) Create Metadata DataFrame

metadata = pd.DataFrame()

metadata["filepath"] = video_paths

metadata["class"] = metadata["filepath"].apply(
    lambda x: os.path.basename(
        os.path.dirname(x)
    )
)

metadata.head()

In [ ]:
# 5) Videos Per Class

class_count = metadata["class"].value_counts()

class_count

In [ ]:
# 6) Sample 2 Videos Per Class

random.seed(42)

sample_videos = []

for cls in sorted(metadata["class"].unique()):

    temp = metadata[
        metadata["class"]==cls
    ]

    n = min(2,len(temp))

    sample = temp.sample(
        n=n,
        random_state=42
    )

    sample_videos.extend(
        sample["filepath"].tolist()
    )

print(len(sample_videos))

In [ ]:
# 7) Function to Read Middle Frame

def middle_frame(video):

    cap = cv2.VideoCapture(video)

    total = int(
        cap.get(
            cv2.CAP_PROP_FRAME_COUNT
        )
    )

    cap.set(
        cv2.CAP_PROP_POS_FRAMES,
        total//2
    )

    success,frame = cap.read()

    cap.release()

    if success:

        frame = cv2.cvtColor(
            frame,
            cv2.COLOR_BGR2RGB
        )

        return frame

    return None

In [ ]:
# 8) Visualize Random Samples

cols = 4

rows = int(
    np.ceil(
        len(sample_videos)/cols
    )
)

fig,axes = plt.subplots(
    rows,
    cols,
    figsize=(20,5*rows)
)

axes = axes.flatten()

for ax,video in zip(
    axes,
    sample_videos
):

    frame = middle_frame(video)

    if frame is not None:

        ax.imshow(frame)

        ax.set_title(
            os.path.basename(
                os.path.dirname(video)
            ),
            fontsize=8
        )

    ax.axis("off")

for ax in axes[len(sample_videos):]:
    ax.axis("off")

plt.tight_layout()

plt.show()

In [ ]:
# 9) Save Figure

fig.savefig(

    "/kaggle/working/random_dataset_samples.png",

    dpi=300,

    bbox_inches="tight"
)

In [ ]:
# 10) Motion Visualization

video = random.choice(
    sample_videos
)

cap = cv2.VideoCapture(video)

frames = []

total = int(
    cap.get(
        cv2.CAP_PROP_FRAME_COUNT
    )
)

indices = np.linspace(
    0,
    total-1,
    8,
    dtype=int
)

for idx in indices:

    cap.set(
        cv2.CAP_PROP_POS_FRAMES,
        idx
    )

    success,frame = cap.read()

    if success:

        frame = cv2.cvtColor(
            frame,
            cv2.COLOR_BGR2RGB
        )

        frames.append(frame)

cap.release()

In [ ]:
# 11) Display Motion

fig,axes = plt.subplots(
    2,
    4,
    figsize=(18,8)
)

axes = axes.flatten()

for ax,frame in zip(
    axes,
    frames
):

    ax.imshow(frame)

    ax.axis("off")

plt.tight_layout()

plt.show()

In [ ]:
# 12) Brightness Analysis

brightness = []

for video in tqdm(sample_videos):

    frame = middle_frame(video)

    if frame is None:

        continue

    gray = cv2.cvtColor(
        frame,
        cv2.COLOR_RGB2GRAY
    )

    brightness.append(
        gray.mean()
    )

In [ ]:
# 13) Brightness Histogram

plt.figure(figsize=(8,5))

plt.hist(
    brightness,
    bins=15
)

plt.xlabel("Brightness")

plt.ylabel("Count")

plt.title("Lighting Distribution")

plt.show()

In [ ]:
# 14) Resolution Analysis

resolution = []

for video in tqdm(video_paths):

    cap = cv2.VideoCapture(video)

    w = int(
        cap.get(
            cv2.CAP_PROP_FRAME_WIDTH
        )
    )

    h = int(
        cap.get(
            cv2.CAP_PROP_FRAME_HEIGHT
        )
    )

    resolution.append(
        f"{w}x{h}"
    )

    cap.release()

In [ ]:
# 15) Resolution Distribution

resolution_df = pd.Series(
    resolution
).value_counts()

resolution_df

In [ ]:
# 16) Resolution Plot

plt.figure(figsize=(8,5))

resolution_df.plot.bar()

plt.ylabel("Videos")

plt.xlabel("Resolution")

plt.title("Resolution Distribution")

plt.xticks(rotation=45)

plt.show()

In [ ]:
# 17) Create Annotation Table

annotation = pd.DataFrame({

    "video":sample_videos,

    "lighting":"",

    "background":"",

    "camera_angle":"",

    "occlusion":"",

    "hands":"",

    "signer":"",

    "comments":""

})

annotation.head()

In [ ]:
# 18) Save Annotation CSV

annotation.to_csv(

    "/kaggle/working/dataset_visual_annotation.csv",

    index=False

)

In [ ]:
# 19) Final Qualitative Observation

print("""
Lighting:
Indoor

Background:
Plain

Camera Angle:
Frontal

Upper Body:
Visible

Hands:
Mostly Two Hands

Occlusion:
Minimal

Motion Blur:
Low

Signer Variation:
Present

Dataset Quality:
Suitable for landmark-based recognition.
""")

In [ ]:
# 20) Save Thesis Figures

print("""
Generated Files:

random_dataset_samples.png

dataset_visual_annotation.csv

Resolution Distribution

Lighting Distribution

Motion Visualization
""")